<a href="https://colab.research.google.com/github/Akomon333/Election-simulation/blob/main/ElectionSimulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [71]:
import numpy as np

In [81]:
group_names = ["Left", "Right", "Far-Right", "Change","Velarian"]
groups_size = np.array([68_400, 72_200 , 22_800, 19_000 , 136_800])
partyA_pref = np.array([1.0, 0, 0, 0, 0])  # RL-controlled party
partyB_pref = 1 - partyA_pref
rng = np.random.default_rng()

num_actions = len(groups_size)
Q = np.zeros(num_actions)
alpha = 0.01
gamma = 0.6
epsilon = 0.4
num_episodes = 100000
num_steps_per_episode = 50

In [77]:
def take_action(partyA_pref, action):
    influence = rng.uniform(0.03, 0.1)
    partyA_pref = partyA_pref.copy()
    if action == 0: # Left
        partyA_pref[0] = np.clip(partyA_pref[0] + influence, 0, 1)
        partyA_pref[1] = np.clip(partyA_pref[1] - influence, 0, 1)
        partyA_pref[2] = np.clip(partyA_pref[2] - influence * rng.uniform(1, 3), 0, 1)
    elif action == 1:  # Right
        partyA_pref[1] = np.clip(partyA_pref[1] + influence, 0, 1)
        partyA_pref[2] = np.clip(partyA_pref[2] + rng.uniform(0.5, 1) * influence, 0, 1)
        partyA_pref[0] = np.clip(partyA_pref[0] - rng.uniform(1, 3) * influence, 0, 1)
        partyA_pref[4] = np.clip(partyA_pref[4] - rng.uniform(1, 3) * influence, 0, 1)
    elif action == 2:  # Far-Right
        partyA_pref[2] = np.clip(partyA_pref[2] + influence, 0, 1)
        partyA_pref[1] = np.clip(partyA_pref[1] + rng.uniform(0.5, 1) * influence, 0, 1)
        partyA_pref[0] = np.clip(partyA_pref[0] - influence * rng.uniform(1.5, 5), 0, 1)
        partyA_pref[4] = np.clip(partyA_pref[4] - rng.uniform(1.5, 5) * influence, 0, 1)
    elif action == 3: # Change
        partyA_pref[3] = np.clip(partyA_pref[3] + influence, 0, 1)
        partyA_pref[4] = np.clip(partyA_pref[4] - rng.uniform(1, 3) * influence, 0, 1)
        partyA_pref[2] = np.clip(partyA_pref[2] - rng.uniform(1, 3) * influence, 0, 1)
        partyA_pref[1] = np.clip(partyA_pref[1] - influence * rng.uniform(1, 2), 0, 1)
        partyA_pref[0] = np.clip(partyA_pref[0] - influence * rng.uniform(0.5, 1), 0, 1)
    elif action == 4: # Velarian
        partyA_pref[4] = np.clip(partyA_pref[4] + influence, 0, 1)
        partyA_pref[3] = np.clip(partyA_pref[3] - rng.uniform(0.5, 3) * influence, 0, 1)
        partyA_pref[2] = np.clip(partyA_pref[2] - rng.uniform(1.5, 5) * influence, 0, 1)
        partyA_pref[1] = np.clip(partyA_pref[1] - rng.uniform(1, 3) * influence, 0, 1)
        partyA_pref[0] = np.clip(partyA_pref[0] - rng.uniform(0.5, 1) * influence, 0, 1)
    partyA_pref[action] = np.clip(partyA_pref[action] + influence, 0, 1)
    return partyA_pref

In [74]:
def simulate_votes(group_sizes, partyA_pref):
    votes_partyA = np.random.binomial(groups_size, partyA_pref)
    votes_partyB = group_sizes - votes_partyA
    return votes_partyA, votes_partyB

In [ ]:
for episode in range(num_episodes):
    partyA_pref_episode = partyA_pref.copy()

    for step in range(num_steps_per_episode):


        if np.random.rand() < epsilon:
            action = np.random.randint(num_actions)
        else:
            action = np.argmax(Q)
        partyA_pref_episode = take_action(partyA_pref_episode, action)
        votes_partyA, votes_partyB = simulate_votes(groups_size, partyA_pref_episode)
        reward = votes_partyA.sum()
        Q[action] = Q[action] + alpha * (reward + gamma * np.max(Q) - Q[action])
    if episode % 10000 == 0:
        print(f"Episode: {episode}")
        best_group = np.argmax(Q)
        print(f"RL candidate should target group: {group_names[best_group]}")
        print(f"Total votes Party A: {votes_partyA.sum() / 1000}k")
        print(f"Total votes Party B: {votes_partyB.sum() / 1000}k")
        print(f"Party A pref: {partyA_pref_episode}")


Current best result for party A:
Party A pref: [0.89105003 0.03349848 0.08657919 0.03407729 0.86485078]
RL candidate should target group: Left
Total votes Party A: 184.337k
Total votes Party B: 134.863k